# Phase 2 — Task 2: Contiguous Imputation (All 6 Baseline Models)

**Protocol (from `phase2-downstream-probes.md`):**
* Reconstructs a missing contiguous block of **20 consecutive timesteps** (20% of the 100-step window).
* Mask start is drawn uniformly at random $t \in [0, 79]$ per sample each epoch.
* Frozen Encoder processes the masked window $\rightarrow$ Freshly initialized `SharedDecoder` reconstructs full window.
* **Loss & Metrics computed STRICTLY on masked 20 timesteps only**:
  $$\mathcal{L}_{\text{impute}} = \frac{1}{|\text{mask}|} \sum_{t \in \text{mask}} \| \hat{x}_t - x_t \|^2$$


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/JEPA_LOB/baselines')
!pip install -q lightning pandas numpy torch
print("✓ Environment and Google Drive ready.")


In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import torch
from downstream_common import (
    MODEL_REGISTRY, STOCKS, LATENT_DIM, set_seed,
    train_imputation_probe
)

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
os.makedirs("downstream_results", exist_ok=True)


In [ ]:
# Run Contiguous Imputation Across All 30 Experiments
impute_results = []
print("=" * 80)
print("  TASK 2: CONTIGUOUS IMPUTATION (Masked MSE & MAE)")
print("=" * 80)

for model_name in MODEL_REGISTRY.keys():
    print(f"\nEvaluating Model: {model_name}")
    for stock in STOCKS:
        t0 = time.time()
        metrics = train_imputation_probe(
            model_name, stock, epochs=50, lr=1e-3, batch_size=256, device=device
        )
        elapsed = time.time() - t0
        print(f"  {model_name:<12} / {stock}: Masked MSE = {metrics['masked_test_mse']:.6f}, Masked MAE = {metrics['masked_test_mae']:.6f} ({elapsed:.1f}s)")
        
        row = {
            'model': model_name,
            'stock': stock,
            'masked_test_mse': metrics['masked_test_mse'],
            'masked_test_mae': metrics['masked_test_mae']
        }
        impute_results.append(row)

df_impute = pd.DataFrame(impute_results)
df_impute.to_csv("downstream_results/imputation_results.csv", index=False)
print("\n✓ Saved: downstream_results/imputation_results.csv")


In [ ]:
# Summary Table
pivot_mse = df_impute.pivot(index='model', columns='stock', values='masked_test_mse')
pivot_mse['Mean Masked MSE'] = pivot_mse.mean(axis=1)
pivot_mse = pivot_mse.sort_values(by='Mean Masked MSE', ascending=True)

print("=" * 80)
print("  TASK 2 SUMMARY: MASKED TEST MSE RANKING")
print("=" * 80)
display(pivot_mse.round(6))
